## 🔍 What is Connect Four?
Connect Four is a two-player connection game where the players take turns placing their pieces in a 7x6 grid. The first player to connect four of their pieces in a row, column, or diagonal wins.

### Requirements:
1. Two players take turns dropping discs into a 7-column, 6-row board
2. A disc falls to the lowest available row in the chosen column
3. The game ends when:
    - A player gets four discs in a row (vertical, horizontal, or diagonal). They win.
    - The board is full. It's a draw.
4. Invalid moves should be rejected clearly:
    - Dropping in a full column.
    - Moving out of turn.
    - Moving after the game is over.

Out of scope: 
- UI support
- Concurrent games
- Move history
- Undo
- Board size configuration

## Core Entities and Relationships

Start by asking yourself: what are the main "things" in this problem? Look for nouns in your requirements and think about what responsibilities each one should have. In Connect Four, a few jump out immediately: the game itself, the board where pieces are placed, and the players making moves.


*A common mistake is putting everything in one giant class or splitting things unnecessarily. Good design means each class has a single, clear job. The Board manages grid state and placement rules. The Game orchestrates turns and win checking. Players are just data—names and which color they're playing.*

# Class Design

## Game

##### What Game must track

- The two players, whose turn it is, and the board
- The game state (in progress, won, draw)
- Who won (if anyone)


##### Approach: Boolean Flags for Game State
Some candidates track game state with separate boolean flags. They add fields like isOver, hasWinner, and isDraw to represent whether the game finished and how it ended.


##### Challenges

The problem is that you're maintaining three coupled boolean flags that must stay synchronized. You can represent invalid states your domain doesn't allow:
- isOver=false, hasWinner=true - someone won but the game isn't over?
- isOver=true, hasWinner=true, isDraw=true - both a win and a draw?
- isOver=false, isDraw=true - it's a draw but the game is still going?

##### Approach: Game State Enum

Instead of multiple booleans, use a single enum that captures exactly the states your domain allows.


## Board

Board owns the grid. It knows where discs are, whether a column has space, how discs "fall," and whether a given move creates four in a row.

##### What Board must track

- Fixed dimensions: number of rows and columns
- The current occupancy of each column (the grid)
- Whether there is at least one empty cell left
- Enough information in the grid to check contiguous discs for a given player

## Player
Player represents one participant in the game. From the requirements, the system only needs two things: a way to distinguish the players and a way to know which disc color each one uses.

##### What Player must track

- A name or ID so the Game can compare players 
- The disc color associated with that player

In [ ]:
class Game:
    - board: Board
    - player1: Player
    - player2: Player
    - currentPlayer: Player
    - state: GameState        // IN_PROGRESS, WON, DRAW
    - winner: Player?

    + Game(player1, player2)
    + makeMove(player, column) -> bool
    + getCurrentPlayer() -> Player
    + getGameState() -> GameState
    + getWinner() -> Player?
    + getBoard() -> Board

class Board:
    - rows: int = 6
    - cols: int = 7
    - grid: DiscColor?[rows][cols]

    + Board()
    + getRows() -> int
    + getCols() -> int
    + canPlace(column) -> bool
    + placeDisc(column, color) -> int
    + isFull() -> bool
    + checkWin(row, column, color) -> bool
    + getCell(row, column) -> DiscColor?

class Player:
    - name: string
    - color: DiscColor

    + Player(name, color)
    + getName() -> string
    + getColor() -> DiscColor

enum GameState:
    IN_PROGRESS
    WON
    DRAW

enum DiscColor:
    RED
    YELLOW